---

## 📚 Tóm Tắt Toàn Bộ Kiến Thức

### Function Calling (Phần 1)
- **Định nghĩa**: LLM nhận diện và gọi tools dựa trên user input
- **Các bước**: Tool định nghĩa → Registry → LLM decision → Execution → Result
- **Thủ công vs OpenAI API**: Manual thì cần regex/rules, OpenAI thông minh hơn

### Vấn Đề Thực Tế (Phần 2)
- **Problems**: Nhiều LLM formats, khó maintain, không có giao thức chung
- **Impacts**: Code duplication, tight coupling, không reusable

### MCP là Gì? (Phần 3)
- **Định nghĩa**: Standard protocol để kết nối LLM với tools
- **Kiến trúc**: Client-Server, JSON-RPC, decoupled
- **Lợi ích**: Unified, standard, easy to maintain, reusable

### MCP Server (Phần 4)
- **Components**: Tools, Resources, Prompts, Sampling
- **Implementation**: Sử dụng `mcp` library từ Anthropic
- **Ví dụ**: Calculator server, Weather server

### LLM + MCP (Phần 5)
- **Client kết nối**: Server discovery, tool listing, tool execution
- **Message protocol**: JSON-RPC requests/responses

### LangChain + MCP (Phần 6)
- **Integration**: MCPToolkit, StructuredTool
- **Workflow**: Agent → Tool List → Tool Call → LLM

### LangGraph + MCP (Phần 7)
- **State machine**: Agent Node → Tool Node → Decision → Loop
- **Multi-turn**: Memory, conversation history, context

---

## 🎯 Key Takeaways

1. **Function Calling** là cách LLM gọi tools - nền tảng
2. **MCP** là giao thức standard để standardize function calling
3. **LangChain** tích hợp MCP → dễ xây dựng agents
4. **LangGraph** tạo complex agentic workflows với state management
5. **Combination** (LLM + MCP + LangGraph) = Powerful agentic systems

---

## 📖 Học Tiếp

- Phần tiếp theo sẽ là **bài tập thực hành** (02_MCP_exercises.ipynb)
- Bạn sẽ tự viết code để:
  - Tạo MCP Server từ scratch
  - Kết nối với LLM API (OpenAI/Claude)
  - Xây dựng agentic chatbot hoàn chỉnh
  - Deploy MCP server thực tế

In [ ]:
## 7.5 Test Agentic Chatbot

import asyncio

# Test agent
async def test_agent():
    print("=" * 60)
    print("TEST AGENTIC CHATBOT")
    print("=" * 60)
    
    test_inputs = [
        "5 cộng 3 bằng bao nhiêu?",
        "Hôm nay ở Hà Nội thời tiết thế nào?"
    ]
    
    for user_input in test_inputs:
        print()
        result = await executor.execute_agent_loop(user_input)
        print(f"\n📊 Final State:")
        print(f"   - Total messages: {len(result['messages'])}")
        print(f"   - Tools used: {result['tools_used']}")
        print("=" * 60)

# Chạy test (asyncio)
try:
    asyncio.run(test_agent())
except RuntimeError:
    # Nếu event loop đang chạy (Jupyter environment)
    print("⚠️  Cannot run async in this environment, showing example output instead:\n")
    
    example_output = """
    ============================================================
    TEST AGENTIC CHATBOT
    ============================================================
    
    👤 User Input: 5 cộng 3 bằng bao nhiêu?
    📊 Starting Agent Loop (max 5 iterations)...
    ============================================================
    
    🔄 Iteration 1:
       Current state - Messages: 1, Tools used: 0
       1️⃣  Agent Node: LLM analyzing...
       2️⃣  Tool Node: Calling calculator with args {'operation': 'add', 'a': 5, 'b': 3}
       ✅ Tool Result: 8
    
    🔄 Iteration 2:
       Current state - Messages: 3, Tools used: 1
       1️⃣  Agent Node: LLM analyzing...
       2️⃣  LLM Decision: Provide final answer
       💬 Final Answer: Đã xử lý request của bạn (sử dụng 1 tools). Kết quả được tính toán dựa trên các tools có sẵn.
    
    📊 Final State:
       - Total messages: 4
       - Tools used: ['calculator']
    ============================================================
    """
    print(example_output)

In [ ]:
## 7.4 Mô Phỏng Agent Execution Loop

class AgentExecutor:
    """
    Mô phỏng việc thực thi agent loop
    (Trong thực tế, LangGraph + LLM API sẽ handle)
    """
    
    def __init__(self, tools: list):
        self.tools = tools
        self.tool_map = {tool.name: tool for tool in tools}
    
    async def execute_agent_loop(self, user_input: str, max_iterations: int = 5):
        """
        Thực thi agent loop:
        1. Gửi user input + tools schema đến LLM
        2. LLM quyết định gọi tool nào
        3. Gọi tool và cập nhật state
        4. Lặp lại cho đến khi LLM quyết định stop
        """
        
        state = {
            "messages": [{"role": "user", "content": user_input}],
            "tools_used": [],
            "iteration": 0
        }
        
        print(f"👤 User Input: {user_input}")
        print(f"📊 Starting Agent Loop (max {max_iterations} iterations)...")
        print("=" * 60)
        
        for iteration in range(max_iterations):
            state["iteration"] = iteration
            
            print(f"\n🔄 Iteration {iteration + 1}:")
            print(f"   Current state - Messages: {len(state['messages'])}, Tools used: {len(state['tools_used'])}")
            
            # Bước 1: Agent node (LLM decision)
            print(f"   1️⃣  Agent Node: LLM analyzing...")
            
            # Demo: LLM quyết định (mô phỏng)
            tool_decision = self.mock_llm_decision(user_input, state)
            
            if tool_decision is None:
                print(f"   2️⃣  LLM Decision: Provide final answer")
                final_answer = self._format_final_answer(user_input, state)
                print(f"   💬 Final Answer: {final_answer}")
                state["messages"].append({"role": "assistant", "content": final_answer})
                return state
            
            tool_name, args = tool_decision
            
            # Bước 2: Tool node
            print(f"   2️⃣  Tool Node: Calling {tool_name} with args {args}")
            result = await self.call_tool(tool_name, args)
            print(f"   ✅ Tool Result: {result}")
            
            # Update state
            state["tools_used"].append(tool_name)
            state["messages"].append({
                "role": "assistant",
                "content": f"[Calling tool {tool_name}]"
            })
            state["messages"].append({
                "role": "tool",
                "content": str(result)
            })
        
        print(f"\n⚠️  Max iterations reached. Returning partial result.")
        return state
    
    def mock_llm_decision(self, user_input: str, state: dict):
        """Mock LLM quyết định gọi tool nào"""
        user_lower = user_input.lower()
        
        if "cộng" in user_lower and "5" in user_lower and "3" in user_lower:
            return ("calculator", {"operation": "add", "a": 5, "b": 3})
        elif "thời tiết" in user_lower and "hà nội" in user_lower:
            return ("get_weather", {"city": "Hanoi"})
        elif "thời tiết" in user_lower and "sài gòn" in user_lower:
            return ("get_weather", {"city": "Ho Chi Minh"})
        else:
            return None  # Stop, provide final answer
    
    async def call_tool(self, tool_name: str, arguments: dict):
        """Gọi tool từ tool_map"""
        if tool_name not in self.tool_map:
            return f"Error: Tool {tool_name} not found"
        
        tool = self.tool_map[tool_name]
        
        try:
            # Gọi LangChain tool
            result = tool.invoke(arguments)
            return result
        except Exception as e:
            return f"Error calling tool: {str(e)}"
    
    def _format_final_answer(self, user_input: str, state: dict) -> str:
        """Format lại câu trả lời cuối cùng"""
        tools_used_str = f" (sử dụng {len(state['tools_used'])} tools)" if state['tools_used'] else ""
        return f"Đã xử lý request của bạn{tools_used_str}. Kết quả được tính toán dựa trên các tools có sẵn."

# Demo: Tạo executor
executor = AgentExecutor(tools=tools)

print("✅ Agent Executor được tạo")
print("""
Agent Execution Workflow:
1. User gửi input
2. Agent Node: LLM phân tích input và tools schema
3. LLM quyết định gọi tool nào (hoặc trả lời trực tiếp)
4. Tool Node: Thực thi tool selected
5. Update state với tool result
6. Loop: Quay lại bước 2 nếu cần (multi-turn)
7. Final: Khi LLM quyết định dừng, trả lời user
""")

In [ ]:
## 7.3 Tạo Agentic Chatbot với LangGraph + MCP

from typing import TypedDict, Annotated, Sequence
from datetime import datetime

# Định nghĩa State cho Agent
class AgentState(TypedDict):
    """State của agentic chatbot"""
    messages: list  # Conversation history
    tools_used: list  # Tools đã sử dụng
    timestamp: str  # Thời gian cuối cùng cập nhật

# Tạo graph cho agentic chatbot
class MCPAgentBuilder:
    """
    Builder class để tạo agentic chatbot với MCP
    """
    
    def __init__(self, tools: list):
        self.tools = tools
        self.graph = None
        self.initial_state = {
            "messages": [],
            "tools_used": [],
            "timestamp": datetime.now().isoformat()
        }
    
    def create_graph(self):
        """Tạo LangGraph workflow"""
        print("🏗️  Xây dựng LangGraph workflow...")
        
        graph_description = """
        Agentic Chatbot Graph Structure:
        ┌─────────────────────────────────────────────────┐
        │           START                                  │
        └──────────────────┬──────────────────────────────┘
                           │
                           ▼
        ┌─────────────────────────────────────────────────┐
        │  Agent Node: LLM nhận messages + tools           │
        │  - Quyết định: gọi tool hay trả lời user?       │
        └──────────────────┬──────────────────────────────┘
                           │
                           ▼
        ┌────────────────────────────────────────────────┐
        │  Tool Node: Gọi tools được chọn                │
        │  - Thực thi tool calls                         │
        │  - Cập nhật state với results                  │
        └──────────────────┬──────────────────────────────┘
                           │
                           ▼
        ┌──────────────────────────────────────────────────┐
        │  Decision: Cần gọi thêm tool không?             │
        │  - Nếu có → Quay lại Agent Node                │
        │  - Nếu không → Đến END                         │
        └──────────────────┬──────────────────────────────┘
                           │
                           ▼
        ┌─────────────────────────────────────────────────┐
        │           END (Return Final Response)           │
        └─────────────────────────────────────────────────┘
        """
        print(graph_description)
        
        return graph_description
    
    def add_tool_schemas(self):
        """Thêm tool schemas vào state"""
        tool_schemas = []
        for tool in self.tools:
            tool_schemas.append({
                "name": tool.name,
                "description": tool.description,
                "schema": tool.args_schema if hasattr(tool, 'args_schema') else {}
            })
        return tool_schemas
    
    def format_tools_for_llm(self):
        """Format tools để gửi cho LLM"""
        formatted = "Available Tools:\n"
        for i, tool in enumerate(self.tools, 1):
            formatted += f"{i}. {tool.name}: {tool.description}\n"
        return formatted

# Tạo agent builder với tools
agent_builder = MCPAgentBuilder(tools=tools)

# Tạo graph
graph_structure = agent_builder.create_graph()

# Hiển thị tool schemas
print("\n📋 Tool Schemas cho LLM:")
tool_schemas = agent_builder.add_tool_schemas()
for schema in tool_schemas:
    print(f"  - {schema['name']}: {schema['description']}")

# Format tools
print("\n📝 Tools Formatted for LLM:")
print(agent_builder.format_tools_for_llm())

In [ ]:
# Cài đặt LangGraph
try:
    from langgraph.graph import StateGraph, END
    from langgraph.prebuilt import ToolNode
    print("✅ LangGraph đã cài đặt")
except ImportError:
    print("⚠️  Cài đặt LangGraph...")
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "langgraph"])
    print("✅ Cài đặt thành công")

**LangGraph** là framework để xây dựng **agentic workflows** với state machines.

### Lợi ích của LangGraph:
- **Explicit state management**: Quản lý state rõ ràng
- **Tool use with MCP**: Tích hợp seamless với MCP tools
- **Loops and branching**: Hỗ trợ complex workflows
- **Memory**: Maintain conversation history
- **Error handling**: Robust error handling

### LangGraph Agentic Loop:
```
State
  ↓
Agent Node (LLM với tools)
  ↓
LLM decides: call tool? continue? stop?
  ↓
If call tool → Tool Call Node
  ↓
Update State with Tool Result
  ↓
Back to Agent Node (loop)
  ↓
If stop → End
```

## 7.2 Cài đặt LangGraph

---

# PHẦN 7: LangGraph + MCP - Xây Dựng Agentic Chatbot

## 7.1 Giới Thiệu LangGraph

In [ ]:
## 6.2 Tạo LangChain Agent với MCP Tools

from langchain_core.tools import Tool, StructuredTool
from typing import Any

# Định nghĩa các tools cho LangChain
def calculator_tool(operation: str, a: float, b: float) -> float:
    """Thực hiện phép toán cơ bản"""
    if operation == "add":
        return a + b
    elif operation == "subtract":
        return a - b
    elif operation == "multiply":
        return a * b
    elif operation == "divide":
        if b == 0:
            return None
        return a / b

def weather_tool(city: str) -> str:
    """Lấy thông tin thời tiết"""
    weather_data = {
        "Hanoi": "20°C, Cloudy",
        "Ho Chi Minh": "28°C, Sunny",
        "Da Nang": "25°C, Partly Cloudy",
    }
    return weather_data.get(city, "City not found")

# Tạo LangChain Tools từ functions
tools = [
    StructuredTool.from_function(
        func=calculator_tool,
        name="calculator",
        description="Thực hiện phép toán: add, subtract, multiply, divide",
        args_schema={
            "type": "object",
            "properties": {
                "operation": {
                    "type": "string",
                    "enum": ["add", "subtract", "multiply", "divide"]
                },
                "a": {"type": "number"},
                "b": {"type": "number"}
            },
            "required": ["operation", "a", "b"]
        }
    ),
    StructuredTool.from_function(
        func=weather_tool,
        name="get_weather",
        description="Lấy thông tin thời tiết của một thành phố",
        args_schema={
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "Tên thành phố"}
            },
            "required": ["city"]
        }
    )
]

print("✅ Đã tạo 2 LangChain Tools:")
for tool in tools:
    print(f"   - {tool.name}: {tool.description}")

# Demo: Tool calling trong LangChain
print("\n📊 Khi LangChain Agent cần gọi tool:")
for tool in tools:
    print(f"\nTool: {tool.name}")
    print(f"Description: {tool.description}")
    print(f"Schema: {tool.args_schema if hasattr(tool, 'args_schema') else 'N/A'}")

In [ ]:
# Kiểm tra và cài đặt dependencies
try:
    from langchain.agents import initialize_agent, AgentType
    from langchain_core.tools import Tool, BaseTool
    print("✅ LangChain dependencies đã cài đặt")
except ImportError:
    print("⚠️  Cài đặt LangChain dependencies...")
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "langchain", "langchain-core"])
    print("✅ Cài đặt thành công")

## 6.1 LangChain + MCP Integration

LangChain cung cấp tích hợp với MCP thông qua **MCPToolkit**

**Workflow:**
```
LangChain Agent
    ↓
Cần gọi tool
    ↓
MCP Client (integrated in LangChain)
    ↓
MCP Server
    ↓
Kết quả
```

**Installation:**
```bash
pip install langchain-core mcp
```

---

# PHẦN 6: MCP + LangChain - Tích Hợp Framework

In [ ]:
# 5.1 Tạo MCP Client để kết nối với Server
import asyncio
from mcp.client import ClientSession
from mcp.client.stdio import StdioClientTransport

# Giả sử MCP Server chạy dưới dạng stdio transport
# (Thực tế sẽ là subprocess hoặc network)

class MCPClientExample:
    """
    Ví dụ về MCP Client
    Trong thực tế, Claude API hoặc integration với LangChain sẽ handle phần này
    """
    
    def __init__(self):
        self.session = None
        self.tools = []
    
    async def initialize(self):
        """Khởi tạo kết nối với MCP Server"""
        # Trong notebook này, ta chỉ demo logic
        # Thực tế sẽ cần subprocess hoặc network connection
        print("🔌 Kết nối đến MCP Server...")
    
    async def list_available_tools(self):
        """Lấy danh sách tools từ server"""
        print("📋 Lấy danh sách tools từ server...")
        # Demo: giả lập lấy tools từ calculator server
        demo_tools = [
            {"name": "add", "description": "Cộng hai số"},
            {"name": "subtract", "description": "Trừ hai số"},
            {"name": "multiply", "description": "Nhân hai số"},
            {"name": "divide", "description": "Chia hai số"},
        ]
        self.tools = demo_tools
        return demo_tools
    
    async def call_tool(self, tool_name: str, arguments: dict):
        """Gọi một tool từ server"""
        print(f"🔧 Gọi tool: {tool_name} với arguments: {arguments}")
        # Demo result
        if tool_name == "add":
            return arguments["a"] + arguments["b"]
        elif tool_name == "subtract":
            return arguments["a"] - arguments["b"]
        elif tool_name == "multiply":
            return arguments["a"] * arguments["b"]
        elif tool_name == "divide":
            if arguments["b"] == 0:
                return "Error: Division by zero"
            return arguments["a"] / arguments["b"]

# Demo: MCP Client usage
print("✅ MCP Client class được tạo")
print("""
Điểm quan trọng về MCP Client:
1. Client kết nối đến MCP Server (qua stdio, network, hoặc process)
2. Client gửi request ListTools để lấy danh sách tools
3. Khi LLM quyết định gọi tool, Client gửi CallTool request
4. Server xử lý và trả kết quả
5. Client trả kết quả cho LLM
""")

# Ví dụ về workflow
print("\n📊 MCP Request-Response Flow:")
print("""
┌────────────────────────────────────┐
│  1. Client: ListTools Request      │
├────────────────────────────────────┤
│  {                                  │
│    "jsonrpc": "2.0",                │
│    "method": "tools/list",          │
│    "id": 1                          │
│  }                                  │
└────────────────────────────────────┘
                 │
                 ▼
┌────────────────────────────────────┐
│  2. Server Response: [add, subtract,│
│     multiply, divide]              │
└────────────────────────────────────┘
                 │
                 ▼
┌────────────────────────────────────┐
│  3. Client: CallTool Request       │
├────────────────────────────────────┤
│  {                                  │
│    "jsonrpc": "2.0",                │
│    "method": "tools/call",          │
│    "id": 2,                         │
│    "params": {                      │
│      "name": "add",                 │
│      "arguments": {"a": 5, "b": 3}  │
│    }                                │
│  }                                  │
└────────────────────────────────────┘
                 │
                 ▼
┌────────────────────────────────────┐
│  4. Server: CallTool Response      │
├────────────────────────────────────┤
│  {                                  │
│    "jsonrpc": "2.0",                │
│    "id": 2,                         │
│    "result": {                      │
│      "content": [{                  │
│        "type": "text",              │
│        "text": "8"                  │
│      }]                             │
│    }                                │
│  }                                  │
└────────────────────────────────────┘
""")

---

# PHẦN 5: Kết Nối LLM với MCP Server

## 5.1 MCP Client - Kết Nối với Server

In [ ]:
from mcp.server import Server
from mcp.types import Tool, TextContent, Resource, ResourceTemplate
import json

# Tạo Weather MCP Server
weather_server = Server("weather-server")

# Mock weather database
weather_database = {
    "Hanoi": {
        "temperature": 20,
        "condition": "Cloudy",
        "humidity": 75,
        "wind_speed": 10,
        "description": "Hôm nay ở Hà Nội là một ngày mây, nhiệt độ 20°C"
    },
    "Ho Chi Minh": {
        "temperature": 28,
        "condition": "Sunny",
        "humidity": 80,
        "wind_speed": 5,
        "description": "Hôm nay ở Sài Gòn là một ngày nắng, nhiệt độ 28°C"
    },
    "Da Nang": {
        "temperature": 25,
        "condition": "Partly Cloudy",
        "humidity": 78,
        "wind_speed": 15,
        "description": "Hôm nay ở Đà Nẵng mây có nắng, nhiệt độ 25°C"
    },
    "Can Tho": {
        "temperature": 26,
        "condition": "Rainy",
        "humidity": 85,
        "wind_speed": 8,
        "description": "Hôm nay ở Cần Thơ có mưa, nhiệt độ 26°C"
    }
}

# Tool 1: Lấy thời tiết của một thành phố
@weather_server.call_tool()
async def call_weather_tool(name: str, arguments: dict):
    if name == "get_weather":
        city = arguments.get("city", "").title()
        if city in weather_database:
            data = weather_database[city]
            result = f"Temperature: {data['temperature']}°C, Condition: {data['condition']}, Humidity: {data['humidity']}%"
            return [TextContent(type="text", text=result)]
        else:
            return [TextContent(type="text", text=f"Weather data for {city} not available")]
    
    elif name == "get_weather_forecast":
        city = arguments.get("city", "").title()
        days = arguments.get("days", 1)
        if city in weather_database:
            current = weather_database[city]
            forecast = f"Forecast for {city} ({days} day(s)): {current['description']}"
            return [TextContent(type="text", text=forecast)]
        else:
            return [TextContent(type="text", text=f"Forecast data for {city} not available")]
    
    elif name == "list_cities":
        cities = list(weather_database.keys())
        return [TextContent(type="text", text=f"Available cities: {', '.join(cities)}")]
    
    else:
        return [TextContent(type="text", text=f"Unknown tool: {name}")]

# Đăng ký các tools
@weather_server.list_tools()
async def list_weather_tools():
    return [
        Tool(
            name="get_weather",
            description="Lấy thông tin thời tiết hiện tại của một thành phố",
            inputSchema={
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "Tên thành phố (ví dụ: Hanoi, Ho Chi Minh, Da Nang)"
                    }
                },
                "required": ["city"]
            }
        ),
        Tool(
            name="get_weather_forecast",
            description="Lấy dự báo thời tiết cho ngày tới",
            inputSchema={
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "Tên thành phố"
                    },
                    "days": {
                        "type": "integer",
                        "description": "Số ngày dự báo (mặc định: 1)",
                        "default": 1
                    }
                },
                "required": ["city"]
            }
        ),
        Tool(
            name="list_cities",
            description="Lấy danh sách các thành phố có thông tin thời tiết",
            inputSchema={
                "type": "object",
                "properties": {}
            }
        ),
    ]

# Resources: Dữ liệu có thể đọc được
@weather_server.list_resources()
async def list_weather_resources():
    """
    Resources là dữ liệu read-only mà client có thể truy cập
    Khác với Tools là cần thực thi
    """
    cities = list(weather_database.keys())
    return [
        Resource(
            uri=f"weather://city/{city.lower()}",
            name=f"Current weather in {city}",
            description=f"Real-time weather data for {city}",
            mimeType="application/json"
        ) for city in cities
    ]

@weather_server.read_resource()
async def read_weather_resource(uri: str):
    """Đọc dữ liệu weather từ resource URI"""
    # Ví dụ: weather://city/hanoi
    if uri.startswith("weather://city/"):
        city_name = uri.replace("weather://city/", "").title()
        if city_name in weather_database:
            data = weather_database[city_name]
            return json.dumps(data, indent=2)
    return "Resource not found"

print("✅ Weather MCP Server được tạo thành công")
print("📋 Các tools:")
print("   - get_weather: Lấy thời tiết hiện tại")
print("   - get_weather_forecast: Dự báo thời tiết")
print("   - list_cities: Danh sách thành phố")
print("📦 Resources đã đăng ký cho tất cả thành phố")

## 4.3 Tạo MCP Weather Server - Ví dụ Phức Tạp Hơn

In [ ]:
from mcp.server import Server
from mcp.types import Tool, TextContent

# Bước 1: Tạo MCP Server instance
server = Server("calculator-server")

# Bước 2: Định nghĩa tool handlers
@server.call_tool()
async def call_calculator(name: str, arguments: dict):
    """
    Handler cho tool call
    Được gọi khi client gọi một tool
    """
    if name == "add":
        result = arguments["a"] + arguments["b"]
        return [TextContent(type="text", text=str(result))]
    
    elif name == "subtract":
        result = arguments["a"] - arguments["b"]
        return [TextContent(type="text", text=str(result))]
    
    elif name == "multiply":
        result = arguments["a"] * arguments["b"]
        return [TextContent(type="text", text=str(result))]
    
    elif name == "divide":
        if arguments["b"] == 0:
            return [TextContent(type="text", text="Error: Cannot divide by zero")]
        result = arguments["a"] / arguments["b"]
        return [TextContent(type="text", text=str(result))]
    
    else:
        return [TextContent(type="text", text=f"Unknown tool: {name}")]

# Bước 3: Đăng ký các tools (Tool Discovery)
@server.list_tools()
async def list_tools():
    """
    Trả về danh sách các tools có sẵn
    Client sẽ gọi hàm này để biết tools nào có thể dùng
    """
    return [
        Tool(
            name="add",
            description="Cộng hai số",
            inputSchema={
                "type": "object",
                "properties": {
                    "a": {"type": "number", "description": "Số thứ nhất"},
                    "b": {"type": "number", "description": "Số thứ hai"}
                },
                "required": ["a", "b"]
            }
        ),
        Tool(
            name="subtract",
            description="Trừ hai số (a - b)",
            inputSchema={
                "type": "object",
                "properties": {
                    "a": {"type": "number", "description": "Số bị trừ"},
                    "b": {"type": "number", "description": "Số trừ"}
                },
                "required": ["a", "b"]
            }
        ),
        Tool(
            name="multiply",
            description="Nhân hai số",
            inputSchema={
                "type": "object",
                "properties": {
                    "a": {"type": "number", "description": "Số thứ nhất"},
                    "b": {"type": "number", "description": "Số thứ hai"}
                },
                "required": ["a", "b"]
            }
        ),
        Tool(
            name="divide",
            description="Chia hai số (a / b)",
            inputSchema={
                "type": "object",
                "properties": {
                    "a": {"type": "number", "description": "Số bị chia"},
                    "b": {"type": "number", "description": "Số chia"}
                },
                "required": ["a", "b"]
            }
        ),
    ]

print("✅ MCP Server được tạo thành công")
print("📋 Các tools đã được định nghĩa:")
print("   - add: Cộng hai số")
print("   - subtract: Trừ hai số")
print("   - multiply: Nhân hai số")
print("   - divide: Chia hai số")

## 4.2 Tạo MCP Server Đơn Giản - Calculator Server

In [ ]:
# Kiểm tra xem mcp library có sẵn không
try:
    import mcp
    print("✅ MCP library đã được cài đặt")
except ImportError:
    print("⚠️  MCP library chưa được cài đặt, sẽ cài đặt...")
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "mcp"])
    import mcp
    print("✅ MCP library đã được cài đặt thành công")

## 4.1 Giới Thiệu MCP SDK

Anthropic cung cấp **mcp** library Python để làm việc với MCP.

**Installation:**
```bash
pip install mcp
```

**Khái niệm chính:**
- **Server**: Implements MCP protocol, exposes tools
- **Client**: Connects to server, calls tools
- **Tools**: Functions that can be called
- **Resources**: Data that can be read
- **Prompts**: Template prompts

## 3.3 MCP vs Function Calling - So Sánh Chi Tiết

| Đặc Tính | Function Calling | MCP |
|---------|------------------|-----|
| **Giao thức** | Riêng cho từng LLM (OpenAI, Claude, etc.) | Unified protocol |
| **Standardization** | Non-standard | Official standard (Anthropic) |
| **Tool Discovery** | LLM cần biết tools trước | Dynamic tool discovery |
| **Decoupling** | LLM và Tools coupling cao | Loose coupling |
| **Scalability** | Khó scale nhiều tools | Dễ scale |
| **Reusability** | Tools khó reuse giữa apps | Tools dễ reuse |
| **Error Handling** | Phải implement riêng | Built-in handling |

## 3.4 MCP Message Protocol - JSON-RPC

MCP sử dụng **JSON-RPC 2.0** để communicate:

```json
// Request từ Client
{
  "jsonrpc": "2.0",
  "id": 1,
  "method": "tools/call",
  "params": {
    "name": "calculator",
    "arguments": {"operation": "add", "a": 5, "b": 3}
  }
}

// Response từ Server
{
  "jsonrpc": "2.0",
  "id": 1,
  "result": {
    "content": [{"type": "text", "text": "8"}]
  }
}
```

---

# PHẦN 4: Tạo MCP Server Đơn Giản từ Scratch

In [ ]:
# Vẽ sơ đồ kiến trúc MCP
from typing import List

mcp_architecture = """
┌───────────────────────────────────────────────────────────────────┐
│                    MCP Architecture Diagram                        │
└───────────────────────────────────────────────────────────────────┘

1. HIGH LEVEL VIEW:
┌──────────────────────────────────────────────────────────────────┐
│                         MCP Client                                 │
│  (Claude API / OpenAI / LangChain / LangGraph)                    │
└────────────────────────────┬─────────────────────────────────────┘
                             │
                     MCP Protocol (JSON-RPC)
                             │
        ┌────────────────────┼────────────────────┐
        │                    │                    │
┌───────▼──────────┐ ┌──────▼──────────┐ ┌──────▼──────────┐
│  MCP Server 1    │ │  MCP Server 2    │ │  MCP Server 3  │
│  (Calculator)    │ │  (Weather)       │ │  (Database)    │
└──────────────────┘ └──────────────────┘ └──────────────────┘


2. DETAILED MESSAGE FLOW:

Client                                          Server
  │                                              │
  ├─────── Initialize Request ────────────────→ │
  │        {type: "initialize", version: ...}  │
  │                                              │
  │ ←────── Initialize Response ──────────────┤ │
  │         {version: ..., capabilities: ...}  │
  │                                              │
  │                                              │
  │  [Client sends tools list request]          │
  ├───── ListTools Request ───────────────────→ │
  │                                              │
  │ ←──── ListTools Response ──────────────────┤ │
  │       [{name: "calculator", schema: ...},   │
  │        {name: "weather", schema: ...}]     │
  │                                              │
  │                                              │
  │  [User asks something, LLM decides to use a tool]
  │  [Client calls the tool]                    │
  ├────── CallTool Request ───────────────────→ │
  │       {tool: "calculator", args: {...}}    │
  │                                              │
  │ ←────── CallTool Response ─────────────────┤ │
  │         {result: "8"}                       │
  │                                              │


3. KEY COMPONENTS:

┌─────────────────────────────────┐
│      MCP Client                 │
├─────────────────────────────────┤
│ • LLM Integration               │
│ • Tool Calling Logic            │
│ • Response Formatting           │
│ • Error Handling                │
└────────────┬────────────────────┘
             │
    ┌────────┴────────┐
    │  MCP Protocol   │
    │  (JSON-RPC)     │
    └────────┬────────┘
             │
┌────────────▼────────────────────┐
│      MCP Server                 │
├─────────────────────────────────┤
│ • Tool Registry                 │
│ • Tool Execution                │
│ • Resource Management           │
│ • Prompt Helpers                │
└─────────────────────────────────┘


4. MCP SERVER CAPABILITIES:

┌─────────────────────────────────┐
│    MCP Server Features          │
├─────────────────────────────────┤
│ 1. Tools                        │
│    - Executable functions       │
│    - Takes input, returns output│
│    - Similar to function calling│
│                                 │
│ 2. Resources                    │
│    - Read-only data             │
│    - Files, templates, etc.     │
│                                 │
│ 3. Prompts                      │
│    - Reusable prompt templates  │
│    - System prompts, examples   │
│                                 │
│ 4. Sampling                     │
│    - Delegate inference to LLM  │
│                                 │
└─────────────────────────────────┘
"""

print(mcp_architecture)

```
Với MCP:
┌──────────────┐                    ┌──────────────┐
│   LLM        │  MCP Protocol      │   MCP Server │
│   Client     │ ──────────────────→│   (Tools)    │
│              │ ←────────────────── │              │
└──────────────┘                    └──────────────┘

Lợi ích:
✅ LLM không cần biết implement chi tiết của tool
✅ Giao thức chung cho tất cả LLM providers
✅ Dễ dàng add/remove/swap tools mà không thay đổi LLM
✅ Có thể reuse MCP servers giữa các applications
✅ Standardized, official support từ Anthropic
```

---

# PHẦN 3: MCP là gì? Kiến Trúc và Khái Niệm

## 3.1 Định Nghĩa MCP

**Model Context Protocol (MCP)** là một giao thức tiêu chuẩn (standard protocol) được Anthropic phát triển để:
1. **Kết nối LLM** với các external tools/resources
2. **Tiêu chuẩn hóa** cách thức communication giữa LLM clients và tool servers
3. **Decoupling** logic của LLM từ logic của tools

**Tương tự như:**
- HTTP là giao thức để web browsers kết nối với web servers
- MCP là giao thức để LLM clients kết nối với tool servers

## 3.2 MCP Architecture - Kiến Trúc

---

# PHẦN 2: Vấn Đề Thực Tế - Tại Sao Cần MCP?

## 2.1 Các Vấn Đề Khi Sử Dụng Function Calling Thông Thường

### Vấn Đề 1: Nhiều LLM, Nhiều Format Khác Nhau
```
OpenAI API:
{
  "type": "function",
  "function": { "name": "...", "parameters": ... }
}

Claude API:
{
  "name": "tool_name",
  "description": "...",
  "input_schema": { ... }
}

Gemini API:
{
  "function_declarations": [
    { "name": "...", "description": "...", "parameters": ... }
  ]
}
```

**Vấn đề**: Mỗi LLM provider có format khác → Phải code riêng cho từng LLM

### Vấn Đề 2: Nhiều Tools, Khó Quản Lý
- System A cần: calculator, weather, search
- System B cần: calendar, email, database_query
- System C cần: calculator, weather, calendar, email, database_query

**Vấn đề**: Nhiều tools được copy-paste, khó maintain, dễ inconsistent

### Vấn Đề 3: Tool Server Vs LLM Coupling
```
Cách thủ công:
┌──────────────┐      request      ┌──────────────┐
│   LLM        │ ─────────────────→ │   Tool       │
│   Client     │ ←───────────────── │   Server     │
└──────────────┘      response      └──────────────┘

Problem:
- LLM cần biết chi tiết của tool
- Thay đổi tool → phải update LLM code
- Khó scale khi có nhiều tool servers
```

### Vấn Đề 4: Không Có Giao Thức Chung
- Mỗi company tự implement cách của riêng
- Không interoperability
- Khó tích hợp tools từ các sources khác nhau

## 2.2 Giải Pháp: Model Context Protocol (MCP)

In [ ]:
# Ví dụ cấu trúc OpenAI Function Calling
# (Không cần API key để xem cấu trúc)

openai_tools_schema = [
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Thực hiện các phép toán cơ bản (cộng, trừ, nhân, chia)",
            "parameters": {
                "type": "object",
                "properties": {
                    "operation": {
                        "type": "string",
                        "enum": ["add", "subtract", "multiply", "divide"],
                        "description": "Phép toán cần thực hiện"
                    },
                    "a": {
                        "type": "number",
                        "description": "Số thứ nhất"
                    },
                    "b": {
                        "type": "number",
                        "description": "Số thứ hai"
                    }
                },
                "required": ["operation", "a", "b"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Lấy thông tin thời tiết của một thành phố",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "Tên thành phố"
                    }
                },
                "required": ["city"]
            }
        }
    }
]

print("📋 OpenAI Function Schema:")
print(json.dumps(openai_tools_schema, indent=2, ensure_ascii=False))

print("\n✅ Lợi ích của OpenAI Function Calling vs cách thủ công:")
print("""
┌─────────────────────────┬────────────────────┬─────────────────────────┐
│ Tiêu chí                │ Cách Thủ Công      │ OpenAI Function Calling │
├─────────────────────────┼────────────────────┼─────────────────────────┤
│ Trích xuất tham số      │ Dùng regex/rule    │ Thông minh, hiểu context │
│ Xử lý lỗi               │ Phải code riêng    │ Tự động                 │
│ Hiểu intent             │ Heuristics đơn     │ Mô hình ngôn ngữ        │
│ Khả năng mở rộng        │ Khó khi thêm tool  │ Dễ dàng thêm tool mới   │
│ Performance             │ Nhanh nhưng giới hạn│ Chậm hơn nhưng smart hơn │
└─────────────────────────┴────────────────────┴─────────────────────────┘
""")

## 1.3 Function Calling với OpenAI API (Cách Chuyên Nghiệp)

Cách làm trên là để hiểu rõ logic. Trong thực tế, ta sử dụng OpenAI Function Calling API:

### Lợi ích của OpenAI Function Calling:
- **LLM thông minh hơn**: OpenAI API tự động phân tích user input để quyết định gọi tool nào
- **Xử lý tham số tốt**: API trích xuất tham số chính xác từ text
- **Handling errors**: API xử lý các trường hợp edge case
- **Streaming**: Support real-time streaming responses

### Cấu trúc OpenAI Function Calling:

In [ ]:
# BƯỚC 6: Test Function Calling
print("=" * 60)
print("TEST 1: Tính toán")
print("=" * 60)
function_calling_loop("5 cộng 3 bằng bao nhiêu?")

print("\n")
print("=" * 60)
print("TEST 2: Hỏi thời tiết")
print("=" * 60)
function_calling_loop("Hôm nay ở Hà Nội thời tiết thế nào?")

print("\n")
print("=" * 60)
print("TEST 3: Tìm kiếm thông tin")
print("=" * 60)
function_calling_loop("Python là gì?")

In [ ]:
# BƯỚC 4: Mô phỏng LLM đưa ra Tool Call Decision
# (Thực tế sẽ dùng OpenAI, Claude API, nhưng ở đây ta mock)

class MockLLMResponse:
    """
    Mô phỏng phản hồi của LLM
    Trong thực tế, LLM sẽ phân tích user input và quyết định gọi tool nào
    """
    def __init__(self, tool_name: str, parameters: Dict):
        self.tool_name = tool_name
        self.parameters = parameters
        self.reasoning = f"Gọi tool '{tool_name}' với tham số {parameters}"

# Mô phỏng: LLM nhận user input và quyết định gọi tool
def mock_llm_decision(user_input: str) -> MockLLMResponse:
    """
    Giả lập LLM phân tích user input và quyết định gọi tool nào
    Trong thực tế, đây sẽ là gọi OpenAI/Claude API
    """
    user_input_lower = user_input.lower()
    
    # Heuristics (rules) để quyết định tool
    if "cộng" in user_input_lower or "cộng" in user_input_lower:
        # User muốn tính 5 + 3
        if "5" in user_input and "3" in user_input:
            return MockLLMResponse("calculator", {"operation": "add", "a": 5, "b": 3})
    
    if "thời tiết" in user_input_lower and "hà nội" in user_input_lower:
        return MockLLMResponse("get_weather", {"city": "Hanoi"})
    
    if "thời tiết" in user_input_lower and "sài gòn" in user_input_lower:
        return MockLLMResponse("get_weather", {"city": "Ho Chi Minh"})
    
    if "python" in user_input_lower or "machine learning" in user_input_lower:
        return MockLLMResponse("search_knowledge_base", {"query": user_input})
    
    return None

# BƯỚC 5: Function Calling Loop - Vòng lặp thực thi tool calls
def function_calling_loop(user_input: str) -> str:
    """
    Vòng lặp function calling:
    1. LLM nhận input
    2. LLM quyết định gọi tool nào
    3. Hệ thống gọi tool
    4. LLM xử lý kết quả
    """
    print(f"👤 User: {user_input}")
    print("-" * 60)
    
    # Bước 1: LLM nhận input và quyết định
    llm_decision = mock_llm_decision(user_input)
    
    if llm_decision is None:
        return "Xin lỗi, tôi không biết cách giúp bạn."
    
    # Bước 2: Thực thi tool call
    print(f"🤖 LLM Decision:")
    print(f"   Tool: {llm_decision.tool_name}")
    print(f"   Parameters: {llm_decision.parameters}")
    print()
    
    try:
        # Gọi tool từ registry
        result = registry.call_tool(llm_decision.tool_name, **llm_decision.parameters)
        print(f"🔧 Tool Execution Result:")
        print(f"   {json.dumps(result, indent=4, ensure_ascii=False)}")
        print()
        
        # Bước 3: LLM xử lý kết quả và trả lời user
        response = format_llm_response(llm_decision.tool_name, result, user_input)
        print(f"💬 Assistant: {response}")
        return response
    
    except Exception as e:
        error_msg = f"Có lỗi khi thực thi tool: {str(e)}"
        print(f"❌ Error: {error_msg}")
        return error_msg

def format_llm_response(tool_name: str, result: Any, user_input: str) -> str:
    """Định dạng kết quả thành câu trả lời cho user"""
    if tool_name == "calculator":
        op = "cộng" if result > 0 else "trừ"
        return f"Kết quả là: {result}"
    
    elif tool_name == "get_weather":
        if "error" in result:
            return result["error"]
        return f"Ở đây thời tiết là {result['condition']}, nhiệt độ {result['temp']}°C, độ ẩm {result['humidity']}%"
    
    elif tool_name == "search_knowledge_base":
        return f"Tìm thấy: {result}"
    
    return f"Kết quả: {result}"

print("✅ Function Calling Loop đã được định nghĩa")

In [ ]:
# BƯỚC 2: Tạo Registry để quản lý tools
class ToolRegistry:
    """
    Quản lý tất cả các tools có sẵn
    Giống như một "công cụ tra cứu" để tìm function cần gọi
    """
    def __init__(self):
        self.tools: Dict[str, Dict[str, Any]] = {}
    
    def register(self, name: str, func: Callable, description: str, parameters: Dict):
        """
        Đăng ký một tool
        Args:
            name: tên của tool (được LLM sử dụng để gọi)
            func: hàm Python thực tế
            description: mô tả tool (cho LLM biết công dụng)
            parameters: schema của các tham số
        """
        self.tools[name] = {
            "func": func,
            "description": description,
            "parameters": parameters
        }
    
    def get_tool_schema(self) -> str:
        """
        Trả về schema của tất cả tools (dùng để gửi cho LLM)
        LLM sẽ dùng schema này để biết tools nào có sẵn
        """
        schema = []
        for name, tool_info in self.tools.items():
            schema.append({
                "name": name,
                "description": tool_info["description"],
                "parameters": tool_info["parameters"]
            })
        return json.dumps(schema, indent=2)
    
    def call_tool(self, tool_name: str, **kwargs) -> Any:
        """
        Gọi một tool bằng tên
        """
        if tool_name not in self.tools:
            raise ValueError(f"Tool không tồn tại: {tool_name}")
        
        func = self.tools[tool_name]["func"]
        return func(**kwargs)

# BƯỚC 3: Đăng ký các tools vào registry
registry = ToolRegistry()

# Đăng ký tool: calculator
registry.register(
    name="calculator",
    func=calculator,
    description="Thực hiện các phép toán cơ bản (cộng, trừ, nhân, chia)",
    parameters={
        "type": "object",
        "properties": {
            "operation": {
                "type": "string",
                "enum": ["add", "subtract", "multiply", "divide"],
                "description": "Phép toán cần thực hiện"
            },
            "a": {
                "type": "number",
                "description": "Số thứ nhất"
            },
            "b": {
                "type": "number",
                "description": "Số thứ hai"
            }
        },
        "required": ["operation", "a", "b"]
    }
)

# Đăng ký tool: get_weather
registry.register(
    name="get_weather",
    func=get_weather,
    description="Lấy thông tin thời tiết của một thành phố",
    parameters={
        "type": "object",
        "properties": {
            "city": {
                "type": "string",
                "description": "Tên thành phố"
            }
        },
        "required": ["city"]
    }
)

# Đăng ký tool: search_knowledge_base
registry.register(
    name="search_knowledge_base",
    func=search_knowledge_base,
    description="Tìm kiếm thông tin từ knowledge base",
    parameters={
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Từ khóa tìm kiếm"
            }
        },
        "required": ["query"]
    }
)

print("✅ Đã đăng ký 3 tools vào registry")
print("\n📋 Schema của tất cả tools:")
print(registry.get_tool_schema())

In [ ]:
# BƯỚC 1: Định nghĩa các tools (công cụ)
import json
from typing import Any, Callable, Dict

# Tool 1: Tính toán đơn giản
def calculator(operation: str, a: float, b: float) -> float:
    """
    Thực hiện phép toán cơ bản
    Args:
        operation: 'add', 'subtract', 'multiply', 'divide'
        a, b: các số cần tính toán
    """
    if operation == "add":
        return a + b
    elif operation == "subtract":
        return a - b
    elif operation == "multiply":
        return a * b
    elif operation == "divide":
        if b == 0:
            raise ValueError("Không thể chia cho 0")
        return a / b
    else:
        raise ValueError(f"Phép toán không hợp lệ: {operation}")

# Tool 2: Lấy thông tin thời tiết (mock)
def get_weather(city: str) -> Dict[str, Any]:
    """Lấy thông tin thời tiết của một thành phố"""
    weather_data = {
        "Hanoi": {"temp": 20, "condition": "Cloudy", "humidity": 75},
        "Ho Chi Minh": {"temp": 28, "condition": "Sunny", "humidity": 80},
        "Da Nang": {"temp": 25, "condition": "Partly Cloudy", "humidity": 78},
    }
    if city not in weather_data:
        return {"error": f"Không có dữ liệu thời tiết cho {city}"}
    return weather_data[city]

# Tool 3: Tìm kiếm thông tin (mock)
def search_knowledge_base(query: str) -> str:
    """Tìm kiếm thông tin từ knowledge base"""
    knowledge = {
        "python": "Python là ngôn ngữ lập trình mạnh mẽ, dễ học",
        "machine learning": "Machine Learning là lĩnh vực AI về máy học từ dữ liệu",
        "llm": "LLM (Large Language Model) là các mô hình ngôn ngữ lớn như GPT, Claude",
    }
    query_lower = query.lower()
    for key, value in knowledge.items():
        if key in query_lower:
            return value
    return f"Không tìm thấy thông tin về: {query}"

print("✅ Đã định nghĩa 3 tools: calculator, get_weather, search_knowledge_base")

## 1.1 Khái Niệm Function Calling / Tool Calling

**Function Calling** (hay **Tool Calling**) là khả năng của LLM để:
1. **Nhận diện** rằng cần gọi một công cụ (tool) nào đó
2. **Chọn tool** phù hợp từ danh sách các tools có sẵn
3. **Trích xuất tham số** từ prompt của user
4. **Thực thi** công cụ với các tham số đó
5. **Trả kết quả** cho LLM để LLM tiếp tục xử lý

### Ví dụ thực tế:
```
User: "Hôm nay ở Hà Nội thời tiết thế nào?"
LLM sẽ:
  → Nhận diện cần dùng tool: get_weather
  → Trích xuất tham số: city="Hanoi"
  → Gọi hàm get_weather(city="Hanoi")
  → Nhận kết quả: "20°C, mây"
  → Trả lời user: "Hôm nay ở Hà Nội 20°C, thời tiết mây..."
```

---

## 1.2 Cách Làm Thủ Công: Xây Dựng Function Calling từ Scratch

# PHẦN 1: Function Calling (Tool Calling) - Cơ Bản từ Scratch

# Model Context Protocol (MCP): Từ Cơ Bản đến Nâng Cao

## 📚 Mục tiêu học tập
- Hiểu rõ khái niệm **Function Calling** / **Tool Calling** từ cơ bản
- Nắm vững vấn đề thực tế khi sử dụng function calling
- Hiểu **Model Context Protocol (MCP)** là gì và tại sao cần nó
- Viết **MCP Server** đầu tiên
- Kết nối **LLM** với **MCP Server**
- Tích hợp **MCP** vào **LangChain** và **LangGraph**

---

## 🎯 Cấu trúc Notebook
1. **Function Calling (Tool Calling) cơ bản**
2. **Vấn đề thực tế - Tại sao cần MCP?**
3. **MCP là gì? Kiến trúc và khái niệm**
4. **Tạo MCP Server đơn giản**
5. **Kết nối LLM với MCP Server**
6. **MCP + LangChain**
7. **MCP + LangGraph (Agentic Chatbot)**